In [5]:
import torch
from transformers import pipeline
import pandas as pd
from tqdm import tqdm
import re

# --- 1. 모델 로드 ---
# Hugging Face 토큰이 필요한 경우 미리 로그인합니다.
# from huggingface_hub import login
# login("YOUR_HF_TOKEN")

print("모델을 로딩합니다...")
device_num = 0
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
pipe = pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16}, # torch_dtype은 model_kwargs 안에 넣는 것이 권장됩니다.
    device_map=f"cuda:{device_num}",
)
print("모델 로딩이 완료되었습니다.")


# --- 2. 데이터 및 프롬프트 준비 ---
try:
    # CSV 파일을 읽어옵니다.
    df = pd.read_csv("/data/2_data_server/cv-07/dice/2025_samsung_challenge/data/test.csv")
    print(f"'{df.shape[0]}'개의 행을 가진 CSV 파일을 성공적으로 읽었습니다.")
except FileNotFoundError:
    print("Error: 'test.csv' 파일을 찾을 수 없습니다. 파일 경로를 확인해주세요.")
    exit()

# 프롬프트 템플릿 정의 (이전과 동일)
prompt_template = """
You are an AI assistant that creates concise, affirmative, and descriptive captions from multiple-choice options.
Your task is to look at each option and turn it into a short, positive caption, as if it were describing a photo.
Completely ignore the structure of the question (e.g., "Which is not...", "What is the purpose...") and only use the content of the options (A, B, C, D) to generate the captions.

### Example 1
Question: Which of the following foods is not present in the image?
A: Pizza
B: Blueberries
C: Salmon
D: Avocado

### Output 1
A: A photo of a pizza.
B: A photo of blueberries.
C: A photo of salmon.
D: A photo of an avocado.

### Example 2
Question: What might be the purpose of the person's workout in the image?
A: Building muscle and strength
B: Practicing for a marathon
C: Training for a cycling race
D: Preparing for a swimming competition

### Output 2
A: A person building muscle and strength.
B: A person practicing for a marathon.
C: A person training for a cycling race.
D: A person preparing for a swimming competition.

### Example 3
Question: Where might the family be headed?
A: To a business meeting
B: To a school
C: To a winter ski resort
D: To a summer vacation spot

### Output 3
A: A family is headed to a business meeting.
B: A family is headed to a school.
C: A family is headed to a winter ski resort.
D: A family is headed to a summer vacation spot.

### Your Task
Question: {question}
A: {option_A}
B: {option_B}
C: {option_C}
D: {option_D}

### Output
"""


def generate_captions_for_row(row, text_generation_pipeline):
    """
    DataFrame의 한 행을 받아 4개의 캡션을 생성하고 리스트로 반환합니다.
    """
    # 프롬프트에 데이터 채우기
    filled_prompt = prompt_template.format(
        question=row['Question'],
        option_A=row['A'],
        option_B=row['B'],
        option_C=row['C'],
        option_D=row['D']
    )
    
    messages = [
        {"role": "user", "content": filled_prompt},
    ]

    try:
        # 모델 호출을 위한 pipeline 설정값
        # max_new_tokens: 생성할 최대 토큰 수
        # temperature: 낮을수록 결정론적이고 일관된 결과 출력 (0.1 ~ 0.3 추천)
        # top_p: 확률이 높은 단어들만 샘플링
        outputs = text_generation_pipeline(
            messages,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.1, 
            top_p=0.9,
        )
        
        # Llama 3.1 Instruct 모델의 응답 부분만 추출
        response_text = outputs[0]["generated_text"][-1]['content'].strip()
        
        # "A:", "B:", ... 부분을 제거하고 텍스트만 추출하여 리스트 생성
        # re.split을 사용하여 "A: ", "B: " 등 다양한 패턴을 기준으로 분리
        captions = re.split(r'\n[A-Z]:\s*', response_text)
        
        # 첫 번째 요소가 "A:"로 시작할 경우, 앞부분을 정리
        if captions[0].startswith('A:'):
             captions[0] = captions[0][3:]

        # 리스트의 첫 요소가 비어있을 경우 제거
        if not captions[0]:
            captions.pop(0)

        # 4개의 캡션이 정확히 생성되었는지 확인
        if len(captions) == 4:
            return captions
        else:
            print(f"\nWarning: 행 ID {row['ID']}에서 4개가 아닌 {len(captions)}개의 캡션이 생성되었습니다. Raw output: '{response_text}'")
            # 부족한 부분을 None으로 채워서 반환
            return (captions + [None] * 4)[:4]

    except Exception as e:
        print(f"\nError: 행 ID {row['ID']} 처리 중 오류 발생: {e}")
        return [None, None, None, None]


# --- 3. 전체 데이터프레임에 대해 캡션 생성 실행 ---
# 결과를 저장할 리스트 초기화
generated_captions = []

# tqdm을 사용하여 진행 상황 표시
print("캡션 생성을 시작합니다...")
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generating Captions"):
    # 각 행에 대해 캡션 생성
    captions = generate_captions_for_row(row, pipe)
    generated_captions.append(captions)

print("캡션 생성이 완료되었습니다.")

# --- 4. 결과 저장 ---
# 생성된 캡션 리스트를 DataFrame으로 변환
captions_df = pd.DataFrame(generated_captions, columns=['caption_A', 'caption_B', 'caption_C', 'caption_D'])

# 기존 DataFrame과 생성된 캡션 DataFrame을 결합
result_df = pd.concat([df, captions_df], axis=1)

# 결과를 새로운 CSV 파일로 저장
output_filename = "test_with_captions.csv"
result_df.to_csv(output_filename, index=False, encoding='utf-8-sig')

print(f"\n모든 작업이 완료되었습니다. 결과가 '{output_filename}' 파일에 저장되었습니다.")
print("\n결과 미리보기:")
print(result_df.head())

모델을 로딩합니다...


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]
Device set to use cuda:0


모델 로딩이 완료되었습니다.
'680'개의 행을 가진 CSV 파일을 성공적으로 읽었습니다.
캡션 생성을 시작합니다...


Generating Captions:   1%|▏         | 10/680 [00:10<11:35,  1.04s/it]You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Generating Captions:  12%|█▏        | 82/680 [01:15<08:04,  1.23it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



A photo of Italian cuisine.
A photo of American cuisine.
A photo of Indian cuisine.'


...


A child is skiing.
A child is playing basketball.
A child is playing soccer.'


Generating Captions:  98%|█████████▊| 668/680 [10:27<00:09,  1.21it/s]Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



A photo of pasta.
A photo of sushi.
A photo of pizza.'


Generating Captions: 100%|██████████| 680/680 [10:37<00:00,  1.07it/s]

캡션 생성이 완료되었습니다.

모든 작업이 완료되었습니다. 결과가 'test_with_captions.csv' 파일에 저장되었습니다.

결과 미리보기:
         ID                     img_path  \
0  TEST_000  ./input_images/TEST_000.jpg   
1  TEST_001  ./input_images/TEST_001.jpg   
2  TEST_002  ./input_images/TEST_002.jpg   
3  TEST_003  ./input_images/TEST_003.jpg   
4  TEST_004  ./input_images/TEST_004.jpg   

                                            Question  \
0  Which of the following foods is not present in...   
1  What might be the purpose of the person's work...   
2                  Where might the family be headed?   
3  Where is the woman likely to be based on the b...   
4            What is the woman in the picture doing?   

                              A                          B  \
0                         Pizza                Blueberries   
1  Building muscle and strength  Practicing for a marathon   
2         To a business meeting                To a school   
3                   In a forest             In a city park   
4  